In [1]:
import numpy as np
import pandas as pd
import sklearn
import nltk
import spacy
import torch
import transformers

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("\nAll libraries loaded successfully!")

numpy: 2.4.4
pandas: 3.0.2
scikit-learn: 1.8.0
torch: 2.11.0+cpu
transformers: 5.8.0

All libraries loaded successfully!


In [2]:
import nltk
nltk.download('reuters')
nltk.download('punkt')
nltk.download('stopwords')

print("download is done")

download is done


[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\spp088\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\spp088\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\spp088\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
from nltk.corpus import reuters
import pandas as pd

# Load documents and their categories
documents = []
for fileid in reuters.fileids():
    category = reuters.categories(fileid)[0] # take first category
    text = reuters.raw(fileid)
    documents.append({'text': text, 'label': category})

df = pd.DataFrame(documents)
print("Total documents:", len(df))
print("\nCategory counts:")
print(df['label'].value_counts().head(10))
df.head()

Total documents: 10788

Category counts:
label
earn            3926
acq             2369
crude            552
interest         453
money-fx         362
trade            329
grain            295
corn             205
dlr              169
money-supply     154
Name: count, dtype: int64


,text,label
0,ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RI...,trade
1,CHINA DAILY SAYS VERMIN EAT 7-12 PCT GRAIN STO...,grain
2,JAPAN TO REVISE LONG-TERM ENERGY DEMAND DOWNWA...,crude
3,THAI TRADE DEFICIT WIDENS IN FIRST QUARTER\n ...,corn
4,INDONESIA SEES CPO PRICE RISING SHARPLY\n Ind...,palm-oil


In [5]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # tokenize
    tokens = word_tokenize(text)
    # remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
print("Cleaning done!")
df[['text', 'clean_text']].head(3)

Cleaning done!


,text,clean_text
0,ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RI...,asian exporters fear damage usjapan rift mount...
1,CHINA DAILY SAYS VERMIN EAT 7-12 PCT GRAIN STO...,china daily says vermin eat 712 pct grain stoc...
2,JAPAN TO REVISE LONG-TERM ENERGY DEMAND DOWNWA...,japan revise longterm energy demand downwards ...


In [7]:
from sklearn.model_selection import train_test_split

# Keep only top 6 categories to keep it manageable
top_categories = df['label'].value_counts().head(6).index
df_filtered = df[df['label'].isin(top_categories)].copy()

print("Working with these categories:", list(top_categories))
print("Dataset size:", len(df_filtered))

X_train, X_test, y_train, y_test = train_test_split(
    df_filtered['clean_text'],
    df_filtered['label'],
    test_size=0.2,
    random_state=42
)

print("\nTrain size:", len(X_train))
print("Test size:", len(X_test))

Working with these categories: ['earn', 'acq', 'crude', 'interest', 'money-fx', 'trade']
Dataset size: 7991

Train size: 6392
Test size: 1599


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Build pipeline: TF-IDF +Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
print("Model Trained!")

# Evaluate
y_pred = pipeline.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Model Trained!

Classification Report:
              precision    recall  f1-score   support

         acq       0.93      0.99      0.96       471
       crude       0.94      0.90      0.92       117
        earn       0.99      0.97      0.98       796
    interest       0.87      0.86      0.86        92
    money-fx       0.84      0.74      0.79        66
       trade       0.95      0.93      0.94        57

    accuracy                           0.95      1599
   macro avg       0.92      0.90      0.91      1599
weighted avg       0.95      0.95      0.95      1599

